### Exploring dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)

print(df.dtypes)

In [ ]:
df.head(2)

In [ ]:
print(df.isnull().sum())

In [ ]:
df[df.duplicated()]

TotalCharges    ->  object (it should be Float64)

In [ ]:
# 1. See the exact Python type of the first few values
print(df['TotalCharges'].apply(type).unique())

### Fix type
<small>TotalCharges : object -> Float64

empty/wrong values -> NaN

In [ ]:
# df['TotalCharges'] = df['TotalCharges'].replace(' ', '0')
# df['TotalCharges'] = df['TotalCharges'].astype('Float64')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df['TotalCharges'].dtype)

### Target Var : Churn

In [ ]:
# Distribution
print(df['Churn'].value_counts(normalize=True))

In [ ]:
df.describe()

In [ ]:
# Visualisation
df['Churn'].value_counts().plot(kind='bar', color=['#534AB7','#E24B4A'])
plt.title('Churn Distribution')
plt.ylabel('Number of customers')
plt.show()

Numerical features

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
# Distributions par Churn
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(num_cols):
    df.groupby('Churn')[col].hist(alpha=0.6, ax=axes[i])
    axes[i].set_title(col)
    axes[i].legend(['No Churn', 'Churn'])
plt.tight_layout()
plt.show()

1. Churners are massively concentrated in the first months (0–10). 
Loyal customers (blue) have a big spike on the right around 70 months.
Conclusion: the more recent a customer is, the more likely they are to leave.

2. The orange is concentrated on the right (70–100$). 
Loyal customers have a massive spike on the left (~$20). 
Conclusion: customers who pay a lot churn more.

3. Orange is concentrated on the left (0–1000€). This is a direct consequence of tenure: churners leave early, so they have not had time to accumulate much.

In [ ]:
# Matrice de corrélation
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm')
plt.show()

Category features

In [ ]:
#checking for outliers
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"{col}: {n_out} outliers")

In [ ]:
# Identifying categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['customerID', 'Churn']]

# Churn rate by category for each column
for col in cat_cols:
    churn_rate = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').mean()
    ).round(3)
    print(f"\n{col}:")
    print(churn_rate.sort_values(ascending=False))

In [ ]:
#keep relevant columns
cat_cols = ['Contract', 'PaymentMethod', 'InternetService', 'TechSupport', 'OnlineSecurity', 'OnlineBackup']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(
        lambda x: (x=='Yes').mean()
    ).reset_index()
    churn_rate.columns = [col, 'churn_rate']
    
    axes[i].bar(churn_rate[col], churn_rate['churn_rate'], 
                color='#534AB7', alpha=0.8)
    axes[i].set_title(f'Churn rate by {col}')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
#create a binary indicator column for automatic payments
df['auto_payment'] = df['PaymentMethod'].isin(
    ['Bank transfer (automatic)', 'Credit card (automatic)']
).astype(int)

In [ ]:
print(df.columns.tolist())

### Insights : Categorical Variables 
<small>A few features stand out clearly: 
- Contract : month-to-month customers churn at 43% vs only 3% for two-year contracts 
- PaymentMethod : electronic check (45%) vs automatic payments (~16%) 
- InternetService : fiber optic looks risky (42%) but it's probably because fiber customers tend to be on month-to-month contracts too

The real high-risk profile seems to be : fiber + no contract + high bill.
The model will need to capture this interaction.

## Summary: Key EDA Findings

1. **Contract type** is by far the strongest signal (43% vs 3%)
2. **tenure** : churners leave in the first 10 months
3. Paying by electronic check is a strong churn signal (45%)
4. TotalCharges and tenure are highly correlated (0.83) (one might be enough)
5. Gender has almost no impact → dropping it
6. No outliers in numerical features
7. 11 missing values in TotalCharges → converted to NaN with pd.to_numeric

In [ ]:
#save 
df.to_csv('data/telco_clean.csv', index=False)

In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="Telco Churn Data", explorative=True)
profile.to_file("output_report.html")